In [92]:
!pip install torch-geometric -q

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from torchvision.models import efficientnet_v2_m, EfficientNet_V2_M_Weights

from skimage.segmentation import slic
from skimage import graph

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool

In [93]:
BASE_PATH = "/kaggle/input/datasets/armanasrul/isic-2019"

train_df = pd.read_csv(f"{BASE_PATH}/ISIC_2019_Training_GroundTruth.csv")
meta_df  = pd.read_csv(f"{BASE_PATH}/ISIC_2019_Training_Metadata.csv")

class_cols = train_df.columns[1:]
NUM_CLASSES = len(class_cols)

train_df["label_idx"] = train_df[class_cols].values.argmax(axis=1)

train_split, val_split = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["label_idx"],
    random_state=42
)

IMAGE_DIR = f"{BASE_PATH}/ISIC_2019_Training_Input/ISIC_2019_Training_Input"

In [94]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [95]:
weights = EfficientNet_V2_M_Weights.IMAGENET1K_V1
cnn = efficientnet_v2_m(weights=weights)

feature_extractor = cnn.features.to(device)

# Freeze CNN
for param in feature_extractor.parameters():
    param.requires_grad = False

feature_extractor.eval()

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth


100%|██████████| 208M/208M [00:01<00:00, 197MB/s]  


Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU(inplace=True)
  )
  (1): Sequential(
    (0): FusedMBConv(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
          (2): SiLU(inplace=True)
        )
      )
      (stochastic_depth): StochasticDepth(p=0.0, mode=row)
    )
    (1): FusedMBConv(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
          (2): SiLU(inplace=True)
        )
      )
      (stochastic_de

In [96]:
transform = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
])

In [97]:
def build_graph(image_path, label):

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        feat_map = feature_extractor(image_tensor)

    feat_map = feat_map.squeeze(0).cpu()   # [C,H,W]
    C, Hf, Wf = feat_map.shape

    image_np = np.array(image.resize((224,224)))

    segments = slic(image_np, n_segments=75, compactness=10, start_label=0)

    seg_resized = np.array(
        Image.fromarray(segments.astype(np.int32)).resize((Wf, Hf), Image.NEAREST)
    )

    seg_resized = torch.tensor(seg_resized)

    num_nodes = seg_resized.max().item() + 1
    node_features = []

    for i in range(num_nodes):
        mask = seg_resized == i
        if mask.sum() == 0:
            node_features.append(torch.zeros(C))
        else:
            region_feat = feat_map[:, mask].mean(dim=1)
            node_features.append(region_feat)

    x = torch.stack(node_features)

    rag = graph.rag_mean_color(image_np, segments)

    edge_index = []
    for edge in rag.edges:
        edge_index.append([edge[0], edge[1]])
        edge_index.append([edge[1], edge[0]])

    edge_index = torch.tensor(edge_index).t().contiguous()

    data = Data(
        x=x,
        edge_index=edge_index,
        y=torch.tensor(label)
    )

    return data

In [98]:
def build_dataset(df):
    graphs = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = os.path.join(IMAGE_DIR, row["image"] + ".jpg")
        if os.path.exists(img_path):
            graphs.append(build_graph(img_path, row["label_idx"]))
    return graphs

train_graphs = build_dataset(train_split)
val_graphs   = build_dataset(val_split)

100%|██████████| 3800/3800 [22:16<00:00,  2.84it/s]


In [99]:
labels = train_split["label_idx"].values

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class Weights:", class_weights)

Class Weights: tensor([ 0.7001,  0.2459,  0.9530,  3.6518,  1.2069, 13.2580, 12.5180,  5.0400],
       device='cuda:0')


In [100]:
class GATNet(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.gat1 = GATConv(in_channels, hidden_channels, heads=4)
        self.gat2 = GATConv(hidden_channels*4, hidden_channels)
        self.lin  = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch):
        x = F.elu(self.gat1(x, edge_index))
        x = F.elu(self.gat2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(x)

In [101]:
model = GATNet(
    in_channels=1280,
    hidden_channels=256,
    num_classes=NUM_CLASSES
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

train_loader = DataLoader(train_graphs, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_graphs, batch_size=8)

In [102]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total Parameters:", total_params)
print("Trainable Parameters:", trainable_params)

Total Parameters: 1579017
Trainable Parameters: 1579017


In [103]:
print(classification_report(all_labels, all_preds))

auc = roc_auc_score(all_labels, all_probs, multi_class="ovr")
print("AUC:", auc)

              precision    recall  f1-score   support

           0       0.39      0.14      0.21       904
           1       0.55      0.90      0.68      2575
           2       0.75      0.01      0.02       665
           3       0.17      0.14      0.15       173
           4       0.32      0.22      0.26       525
           5       0.50      0.04      0.08        48
           6       0.00      0.00      0.00        51
           7       0.00      0.00      0.00       126

    accuracy                           0.51      5067
   macro avg       0.33      0.18      0.18      5067
weighted avg       0.49      0.51      0.42      5067



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


NameError: name 'all_probs' is not defined

In [ ]:
def evaluate(loader):

    model.eval()

    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            out = model(data.x, data.edge_index, data.batch)
            probs = torch.softmax(out, dim=1)
            preds = probs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(data.y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")

    try:
        auc = roc_auc_score(
            all_labels,
            all_probs,
            multi_class="ovr",
            average="macro"
        )
    except:
        auc = 0

    return acc, precision, recall, f1, auc

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for data in train_loader:
        data = data.to(device)

        optimizer.zero_grad()

        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    acc, prec, rec, f1, auc = evaluate(val_loader)

    print(f"\nEpoch {epoch+1}")
    print("Loss:", round(avg_loss,4))
    print("Accuracy:", round(acc,4))
    print("Precision:", round(prec,4))
    print("Recall:", round(rec,4))
    print("F1:", round(f1,4))
    print("AUC:", round(auc,4))